# T13 — Nhánh đối chứng tiếng Anh CÙNG SỐ TOKEN · Nhóm 8 · chạy trên Google Colab

Repo: https://github.com/KienNguyenDev2711/Hyena-Attention-Study

**Vì sao có notebook này.** Báo cáo hiện viết "nhánh tiếng Anh được cắt xuống đúng cùng số
token với nhánh tiếng Việt", nhưng số thật là VI = 38.250.964 còn EN = 42.147.057 token
(cả hai < ngân sách 50M nên lệnh cắt cũ không bao giờ chạy — xem `docs/04_task_sua_bao_cao.md`,
mục T1/T13). Notebook này chạy lại 6 lần `E1_en` với corpus **cắt xuống đúng 38.250.964
token** trong khi ngân sách huấn luyện vẫn 50M → cùng số bước, cùng ~1,31 epoch với nhánh VI.

| | |
|---|---|
| Số lần chạy | 6 = 2 mô hình (HHHH, AAAA) × 3 seed (0, 1, 2) |
| Ngân sách | ~1 giờ GPU T4 (lần chạy đầu +10 phút token hoá 90k bài, sau đó dùng cache) |
| Sản phẩm | `results_t13/E1_en_*.json` + `*_history.csv`, gói trong `E1_en_matched_results.zip` |

**Trước khi chạy:** Runtime → Change runtime type → chọn **GPU (T4)**.

**Quy tắc của nhóm:** notebook KHÔNG chứa logic thí nghiệm — chỉ clone repo và gọi
`hyena_study.train` (module đã có test đường dây).


## 1 · Nạp mã nguồn

In [ ]:
REPO_URL = "https://github.com/KienNguyenDev2711/Hyena-Attention-Study.git"
WORK     = "/content/Hyena-Attention-Study"

import os, shutil, subprocess, sys

if os.path.isdir(WORK):
    shutil.rmtree(WORK)          # chạy lại từ đầu -> luôn lấy bản mới nhất
subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, WORK], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)

print("Đã clone vào", WORK)


## 2 · Cài thư viện và kiểm tra GPU

In [ ]:
!pip install -q datasets tokenizers

import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} · {p.total_memory/2**30:.1f} GB")
else:
    print("\n" + "!" * 70)
    print("!! CHƯA BẬT GPU — Runtime -> Change runtime type -> GPU (T4), rồi chạy lại")
    print("!! 6 lần chạy 50M token trên CPU là bất khả thi")
    print("!" * 70)


## 3 · Cổng go/no-go TRƯỚC KHI tiêu quota GPU

Hai bước kiểm, dừng lại ngay nếu bước nào kêu:

1. Cờ `--max_train_tokens` phải tồn tại trên bản GitHub — nếu chưa, commit T13 chưa được
   push, chạy tiếp chỉ đốt GPU cho một cấu hình sai.
2. Bộ test đường ống chạy trên CPU (~1 phút) — có FAIL thì mọi kết quả sau đó không đáng tin.


In [ ]:
import subprocess, sys

help_text = subprocess.run(
    [sys.executable, "-m", "hyena_study.train", "--help"],
    capture_output=True, text=True).stdout
assert "--max_train_tokens" in help_text, (
    "Bản trên GitHub CHƯA có cờ --max_train_tokens. "
    "Push commit T13 (nhánh quang/t8-t9-t13 hoặc main) lên GitHub rồi chạy lại từ ô clone."
)
print("OK: cờ --max_train_tokens có mặt.")


In [ ]:
!python tests/test_pipeline.py


## 4 · Sáu lần chạy E1_en cùng số token

Tham số giống hệt E1 gốc (`n_docs 90000`, `token_budget 50000000`, vocab/seq_len/batch/lr
theo default), chỉ thêm `--max_train_tokens 38250964`. Ghi vào `results_t13/` để không
lẫn với kết quả cũ; tên run giữ nguyên `E1_en_*` để khi nhóm chốt thì chép đè vào `results/`.

Lần chạy đầu tốn ~10 phút token hoá 90.000 bài; các lần sau dùng cache `data_cache/`.
Mỗi lần chạy ~6–9 phút trên T4. Chạy tuần tự từng ô, có thể chạy lại riêng ô nào lỗi.


In [ ]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 90000 --token_budget 50000000 --max_train_tokens 38250964 \
        --seed 0 --run_name E1_en_HHHH_s0 --out_dir results_t13


In [ ]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 90000 --token_budget 50000000 --max_train_tokens 38250964 \
        --seed 1 --run_name E1_en_HHHH_s1 --out_dir results_t13


In [ ]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 90000 --token_budget 50000000 --max_train_tokens 38250964 \
        --seed 2 --run_name E1_en_HHHH_s2 --out_dir results_t13


In [ ]:
!python -m hyena_study.train --layers AAAA --lang en --tokenizer syllable \
        --n_docs 90000 --token_budget 50000000 --max_train_tokens 38250964 \
        --seed 0 --run_name E1_en_AAAA_s0 --out_dir results_t13


In [ ]:
!python -m hyena_study.train --layers AAAA --lang en --tokenizer syllable \
        --n_docs 90000 --token_budget 50000000 --max_train_tokens 38250964 \
        --seed 1 --run_name E1_en_AAAA_s1 --out_dir results_t13


In [ ]:
!python -m hyena_study.train --layers AAAA --lang en --tokenizer syllable \
        --n_docs 90000 --token_budget 50000000 --max_train_tokens 38250964 \
        --seed 2 --run_name E1_en_AAAA_s2 --out_dir results_t13


## 5 · Kiểm tra sản phẩm và đóng gói

Kiểm đúng điều kiện T13 đòi hỏi: mỗi file phải ghi `corpus.n_tokens_train == 38.250.964`.


In [ ]:
import json, glob

files = sorted(glob.glob("results_t13/E1_en_*.json"))
assert len(files) == 6, f"kỳ vọng 6 file JSON, thấy {len(files)}: {files}"
print(f"{'run':24s} {'train tokens':>14s} {'tokens seen':>13s} {'test PPL':>9s}")
for f in files:
    d = json.load(open(f))
    n = d["corpus"]["n_tokens_train"]
    assert n == 38250964, f"{f}: n_tokens_train={n} != 38250964 — cờ cắt không ăn!"
    print(f"{d['run_name']:24s} {n:>14,d} {d['tokens_seen']:>13,d} {d['test_ppl']:>9.3f}")
print("\nOK: cả 6 lần chạy đều cắt corpus đúng 38.250.964 token.")


In [ ]:
!zip -qr E1_en_matched_results.zip results_t13
try:
    from google.colab import files
    files.download("E1_en_matched_results.zip")
except Exception as e:
    print("Không tự tải được (", e, ")")
    print("-> Tải tay: panel Files bên trái -> E1_en_matched_results.zip")
